[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/example/shortcut_toy.ipynb)


### A Working `Shortcut` Example

The following code reproduces the results from the toy example reported in the perspective article "Shortcut Learning in Deep Neural Networks" by Robert Geirhos, Jörn-Henrik Jacobsen, Claudio Michaelis, Richard Zemel, Wieland Brendel, Matthias Bethge & Felix A. Wichmann.

### 1) Setting up the environment

The first cells get our workspace ready: load the libraries, mount Google Drive, and download the data files this example depends on.

**Loading our tools.** This cell brings in the libraries the rest of the notebook depends on. `numpy` holds our images as plain grids of numbers; `torch` and its helpers (`nn`, `nn.functional`, `torch.utils.data`) build and train the network; `matplotlib` produces the plots.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import torch.utils.data as utils
import torch.nn as nn
import torch.nn.functional as F
import sys
import IPython
import time
import IPython.display as display

**A few more tools.** `make_grid` arranges many small images into one tidy montage so we can preview a batch at a glance. `cv2` (OpenCV) is what we'll use to *move* a shape around inside an image — that sliding motion is the heart of how we build the datasets.

In [ ]:
from torchvision.utils import make_grid as make_grid
#from PIL import Image
import requests
from io import BytesIO
import cv2

**Connecting Google Drive.** This is a standard Colab step that links your Google Drive to the notebook. This particular example doesn't read or write anything to your Drive, so it isn't strictly required here — but it's a common habit, and harmless to run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**Downloading the data files.** Our two starting shapes (`star.npy` and `moon.npy`) and the figure image live in a GitHub repository. This cell copies that repository into the Colab workspace so the later cells can find those files. The `if not os.path.exists(...)` guard means that if you run it twice, it won't waste time re-downloading.

In [ ]:
import os

repo_url = "https://github.com/geraldmc/irilab2026.git"
repo_name = "irilab2026" # The folder created when you clone

if not os.path.exists(repo_name):
    print(f"Cloning {repo_name}...")
    !git clone $repo_url
else:
    print(f"{repo_name} already exists. Skipping clone.")

In [ ]:
# 1. If necessary, remove the existing repository folder 
# (replace 'repo-name' with your actual folder name) and rerun above cell.
!rm -rf irilab2026

### 2) The shortcut, in pictures

Before writing any code, it helps to see the whole experiment laid out. The figure below is the claim we're about to reproduce ourselves.

When trained on a simple dataset of stars and moons (training set below, tan background), a standard neural network (three layers, fully-connected) can easily categorize novel similar exemplars (middle row, blue background) but testing it on a slightly different dataset (bottom row, red background) reveals a shortcut classification strategy: The network has learned to associate object location with a label. 

During training, stars were always shown in the top right or bottom left of an image; moons in the top left or bottom right, thus the neural network used location instead of shape for categorization. This pattern is still present in samples from the i.i.d. test set (middle row) but no longer present in o.o.d. test images (bottom row). Neural networks often rely on such unintended strategies to solve problems.

**The figure we're reproducing.** This displays the diagram from the paper. Keep it nearby as you read on: everything we build is an attempt to recreate this result from scratch and watch it happen.

In [ ]:
from IPython.display import Image, display
display(Image(filename='/content/irilab2026/example/IMAGE1.png', width=800))

### 3) Configuration

A single place to set the numbers that control the run. Gathering them here, rather than scattering them through the code, makes the notebook easier to adjust and to reproduce.

**The knobs for this run.** The settings that actually matter below are `lr` (the learning rate — how big a step the network takes when it adjusts itself), `bs` (batch size — how many images it looks at before each adjustment), `epochs` (how many full passes it makes through the data), and `log_interval` (how often it prints progress).

In [ ]:
# Train specification
epochs = 100
lr = 0.0001
bs = 100
log_interval = 10

### 4) Building two datasets

This is the core idea, so it's worth stating plainly. We build **two** collections of images from the same two shapes:

- a **biased** set, where each shape only ever appears in its own assigned corners — so an image's *position* perfectly predicts its label, and
- an **unbiased** set, where each shape can appear anywhere — so position tells you nothing.

We will **train on the biased set** and **test on the unbiased set**. That single choice is the entire experiment. If the network truly learned to recognize the *shape*, it will stay accurate on the unbiased set. If it secretly learned the easier rule — *where* the shape sits — it will do well in training and then fall apart on the unbiased set, because the position clue is gone. The gap between those two scores is what we're measuring.

*A note on reproducibility:* the positions are chosen at random and we don't fix a random seed, so the exact images differ every time you run this. The overall outcome — high training accuracy, low test accuracy — does not.

**Loading the two starting shapes.** Each `.npy` file is a single 200×200 image: one star, one moon, drawn in white on a black background. These two pictures are the only raw material we have. Everything else — thousands of training and test images — is made by taking these two shapes and sliding them to different positions.

In [ ]:
moon = np.load("/content/irilab2026/example/moon.npy")
moon = moon

star = np.load("/content/irilab2026/example/star.npy")
star = star

**A quick look, and a preview of the trick.** `cv2.warpAffine` shifts an image by an amount given in the `translation_matrix`. Here the matrix is set to shift by *zero*, so this cell simply displays the star exactly as loaded — a sanity check that the file is what we expect. In the next cells we'll fill that matrix with real shift amounts to *move* the shape around the canvas. (The comment about corners is a note about those upcoming shifts, not about this cell.)

In [ ]:
num_rows, num_cols = moon.shape[:2]

# bottom right: 46 39 to 60 70
translation_matrix = np.float32([ [1,0,0], [0,1,0] ])
img_translation = cv2.warpAffine(star, translation_matrix, (num_cols, num_rows))
plt.imshow(img_translation,cmap='gray')

**Building the biased set.** This loop creates the training images where the shortcut is available. For each shape, the random shift amounts are restricted to a narrow range, so the moon only lands in its two assigned corners and the star only lands in its own two — matching the arrangement in the figure above. We make 2,000 of each. Because every star sits in star-corners and every moon in moon-corners, *position alone* is enough to tell them apart. That's the trap we're deliberately setting.

In [ ]:
biased_train_moon = []
biased_train_star = []

for i in range(1000):
  
  x_val = np.random.randint(46,60)
  y_val = np.random.randint(39,70)
  translation_matrix = np.float32([ [1,0,x_val], [0,1,y_val] ])
  im = cv2.warpAffine(moon, translation_matrix, (200, 200))
  biased_train_moon.append(im)
  x_val = np.random.randint(-53,-39)
  y_val = np.random.randint(-65,-34)
  translation_matrix = np.float32([ [1,0,x_val], [0,1,y_val] ])
  im = cv2.warpAffine(moon, translation_matrix, (200, 200))
  biased_train_moon.append(im)
  
  y_val = np.random.randint(44,64)
  x_val = np.random.randint(-46,-22)
  translation_matrix = np.float32([ [1,0,x_val], [0,1,y_val] ])
  im = cv2.warpAffine(star, translation_matrix, (200, 200))
  biased_train_star.append(im)
  y_val = np.random.randint(-56,-33)
  x_val = np.random.randint(55,78)
  translation_matrix = np.float32([ [1,0,x_val], [0,1,y_val] ])
  im = cv2.warpAffine(star, translation_matrix, (200, 200))
  biased_train_star.append(im)

**Building the unbiased set.** Same two shapes, but now the random shifts cover the *whole* canvas, so a star or a moon can show up anywhere. The position clue is gone — the only honest way to tell these apart is by their shape. The last four lines convert our Python lists into NumPy arrays so they're ready for PyTorch.

*Why only two datasets when the figure shows three rows?* The figure has a training row, a middle 'i.i.d.' test row, and a bottom 'o.o.d.' test row. The middle row is just more biased-style data, included to show the network looks fine on familiar-looking examples. The decisive comparison is training (biased) versus the bottom row (unbiased), so this notebook builds those two and skips constructing the middle row as a separate thing.

In [ ]:
unbiased_train_moon = []
unbiased_train_star = []

for i in range(2000):
  
  x_val = np.random.randint(-53,60)
  y_val = np.random.randint(-65,70)
  translation_matrix = np.float32([ [1,0,x_val], [0,1,y_val] ])
  im = cv2.warpAffine(moon, translation_matrix, (200, 200))
  unbiased_train_moon.append(im)
  
  y_val = np.random.randint(-56,64)
  x_val = np.random.randint(-46,78)
  translation_matrix = np.float32([ [1,0,x_val], [0,1,y_val] ])
  im = cv2.warpAffine(star, translation_matrix, (200, 200))
  unbiased_train_star.append(im)
  
biased_train_star = np.array(biased_train_star)
biased_train_moon = np.array(biased_train_moon)
unbiased_train_star = np.array(unbiased_train_star)
unbiased_train_moon = np.array(unbiased_train_moon)

**Friendlier names, and a preview sample.** The first lines just give the arrays shorter, clearer names. The last two lines pull 25 images aside to display as a montage in the next cell, so you can see what the data actually looks like.

In [ ]:
unbiased_stars = unbiased_train_star
unbiased_moons = unbiased_train_moon
biased_stars = biased_train_star
biased_moons = biased_train_moon

some_unbiased_stars = biased_stars[:25]
np.random.shuffle(some_unbiased_stars)

**Packaging the data for training.** A few things happen here. First we build and save that 25-image montage. Then we assemble the real datasets: we stack the stars and moons together and create matching **labels** — `0` for every star, `1` for every moon. Finally we wrap each dataset in a `DataLoader`, PyTorch's conveyor belt that feeds images to the network in batches of 100 (and shuffles the training data so it doesn't see them in a fixed order).

Notice which data goes where: `train_loader` carries the **biased** images, `test_loader` carries the **unbiased** ones. This is the moment the experiment is locked in.

In [ ]:
grid = make_grid(torch.from_numpy(np.float32(some_unbiased_stars)).view(25,1,200,200),nrow=5,pad_value=1,padding=10)
plt.axis('off')
plt.imsave('biased_stars.png',grid.numpy().transpose(1,2,0))
plt.imshow(grid.numpy().transpose(1,2,0))

unbiased_data = np.concatenate((unbiased_stars,unbiased_moons))
unbiased_data = torch.from_numpy(np.float32(unbiased_data))
labels = np.concatenate((np.zeros(len(unbiased_stars)),np.ones(len(unbiased_stars))))
labels = torch.from_numpy(labels).long()
biased_data = np.concatenate((biased_stars,biased_moons))
biased_data = torch.from_numpy(np.float32(biased_data))

train_data = utils.TensorDataset(biased_data,labels) 
train_loader = utils.DataLoader(train_data,batch_size=100,shuffle=True) 
test_data = utils.TensorDataset(unbiased_data,labels) 
test_loader = utils.DataLoader(test_data,batch_size=100)

### 5) The model

Now we define the network itself — the thing that will try to learn the star-versus-moon distinction.

**Defining the network.** Three network designs are written here, but only the first, `Net`, is actually used. It's a *fully-connected* network: it flattens each 200×200 image into one long list of 40,000 numbers, passes it through two hidden layers, and produces two scores — one for 'star', one for 'moon'. The larger score wins. (`ConvNet` and `ConvNet2` are alternative designs left in for reference; this run doesn't use them.)

In [ ]:
# Build FC net
class Net(nn.Module): # `nn` is the heart of Pytorch
    def __init__(self): # overriding the __init__ constructor
        super(Net, self).__init__() 
        self.fc1 = nn.Linear(200*200, 1024) # layer 1 w/ 200*200 features (the number of pixels) outputting 1024
        self.fc2 = nn.Linear(1024, 1024) # layer 2 w/ 1024 features, outputting 1024 features
        self.fc3 = nn.Linear(1024, 2) # layer 3 w/ 1024 features, outputting 2 for binary prediction (star/moon)
    def forward(self, x): # overriding the forward method
        """ Where __init__ created the parts (the three layers), forward says how data actually flows 
        through them — it maps one-to-one onto the dimension diagram drawn below. Each line is one 
        step down that pipeline."""
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
from IPython.display import Image, display
display(Image(filename='/content/irilab2026/example/fc_net.png', width=550))

In [ ]:
# These two classes are not used yet. 
class ConvNet(nn.Module): # `nn` is the heart of Pytorch
    def __init__(self):
        super(ConvNet, self).__init__()
        self.c1 = nn.Conv2d(1, 32, 5, 1, padding=2)
        self.c2 = nn.Conv2d(32, 32, 5, 1, padding=2)
        self.c3 = nn.Conv2d(32, 32, 5, 1, padding=2)
        self.pool = nn.AvgPool2d(200)
        self.fc = nn.Linear(32, 2)
    def forward(self, x):
        x = F.relu(self.c1(x))
        x = F.relu(self.c2(x))
        x = F.relu(self.c3(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x
      
      
class ConvNet2(nn.Module):
    def __init__(self):
        super(ConvNet2, self).__init__()
        self.c1 = nn.Conv2d(1, 128, 5,stride=2,padding=2)#, 2, padding=2)
        self.c2 = nn.Conv2d(128, 128, 5,stride=2,padding=2)#, 2, padding=2)
        self.c3 = nn.Conv2d(128, 128, 5,stride=2,padding=2)#, 2, padding=2)
        self.pool = nn.AvgPool2d(25)
        self.fc = nn.Linear(128, 2)
    def forward(self, x):
        x = F.relu(self.c1(x))
        x = F.relu(self.c2(x))
        x = F.relu(self.c3(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

### 6) Training, and the reveal

With the data and the network ready, **we train on the biased set and then test on the unbiased set**. The two accuracy numbers at the end are the result.

**Getting ready to train.** We create the network, choose **Adam** as the optimizer (the procedure that adjusts the network's internal numbers to reduce mistakes), and pick **cross-entropy loss** as the score that measures how wrong each prediction is. Training works by nudging the network to make that score smaller.

*Note:* `epochs` is set to 5. The learning rate is set to `lr = 0.0001`. This represents a safe, stable step size to prevent overshooting.

In [ ]:
net = Net()
optimizer = torch.optim.Adam(net.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()
epochs = 5

**Training, then the test.** The first loop is the training loop. For each epoch (one full pass over the biased data), and for each batch within it, the network: flattens the images, makes a guess, measures how wrong it was, and adjusts itself slightly to do better. Along the way it tracks training accuracy.

The second loop runs the trained network on the **unbiased** test data and reports its accuracy there. This is the moment of truth. You'll typically see high training accuracy and much lower test accuracy — the network scored well while the position clue was present, then stumbled once it was removed. That gap is the shortcut, caught in the act: the network learned *where* the shapes were, not *what* they were.

In [ ]:
for epoch in range(epochs):
    total = 0.
    correct = 0.
    for batch_idx, (data, target) in enumerate(train_loader):
        data = data.view(-1, 200*200)
        target = target.squeeze()
        optimizer.zero_grad()
        net_out = net(data).squeeze()
        loss = criterion(net_out, target)
        loss.backward()
        optimizer.step()
        if batch_idx % log_interval == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                    epoch+1, batch_idx * len(data), len(train_loader.dataset),
                           bs * batch_idx / len(train_loader), loss.item()))
            
        _, predicted = torch.max(net_out.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()

print('Train Acc: '+str(correct/total))

total = 0.
correct = 0.
for idx, (data, target) in enumerate(test_loader):
  data = data.view(-1, 200*200)
  optimizer.zero_grad()
  net_out = net(data).squeeze()
  _, predicted = torch.max(net_out.data, 1)
  total += target.size(0)
  correct += (predicted == target).sum().item()
print('Test Acc: '+str(correct/total))